In [ ]:
import pandas as pd
import numpy as np
import json
import re
from pathlib import Path

print("Environment ready")

In [ ]:
import pandas as pd
from pathlib import Path
import time

# Adjust this if your dataset folder is somewhere else relative to the notebook
PROJECT_ROOT = Path.cwd().parents[2]

DATA_DIR = PROJECT_ROOT / "dataset" / "train"

files_to_convert = {
    "train_source1": DATA_DIR / "train_source1.tsv",
    "train_source2": DATA_DIR / "train_source2.tsv",
    "train_source3": DATA_DIR / "train_source3.tsv",
    "train_ground_truth": DATA_DIR / "train_ground_truth.tsv",
}

for name, tsv_path in files_to_convert.items():
    parquet_path = tsv_path.with_suffix(".parquet")

    if parquet_path.exists():
        print(f"Skipping {name} — parquet already exists at {parquet_path}")
        continue

    print(f"Converting {name}...")
    start = time.time()

    df = pd.read_csv(
        tsv_path,
        sep="\t",
        dtype=str,
        usecols=["entity_id", "business_name", "business_address", "country"]
        if "ground_truth" not in name
        else None,  # ground truth has different columns, so don't restrict
    )

    df.to_parquet(parquet_path, index=False)

    elapsed = time.time() - start
    print(f"  Saved {parquet_path} ({len(df):,} rows) in {elapsed:.1f}s")

print("\nDone. From now on, load with pd.read_parquet(...) instead of pd.read_csv(...).")

Converting train_source1...
  Saved c:\Personal\Sachita\business-entity-resolution\dataset\train\train_source1.parquet (2,206,821 rows) in 9.4s
Converting train_source2...
  Saved c:\Personal\Sachita\business-entity-resolution\dataset\train\train_source2.parquet (5,034,616 rows) in 24.5s
Converting train_source3...
  Saved c:\Personal\Sachita\business-entity-resolution\dataset\train\train_source3.parquet (5,285,603 rows) in 24.0s
Converting train_ground_truth...
  Saved c:\Personal\Sachita\business-entity-resolution\dataset\train\train_ground_truth.parquet (2,206,821 rows) in 5.3s

Done. From now on, load with pd.read_parquet(...) instead of pd.read_csv(...).


In [4]:
import time

PROJECT_ROOT = Path.cwd().parents[2]
DATA_DIR = PROJECT_ROOT / "dataset" / "train"

print("Loading datasets from parquet...")
start = time.time()

source1 = pd.read_parquet(DATA_DIR / "train_source1.parquet")
source2 = pd.read_parquet(DATA_DIR / "train_source2.parquet")
source3 = pd.read_parquet(DATA_DIR / "train_source3.parquet")
ground_truth = pd.read_parquet(DATA_DIR / "train_ground_truth.parquet")

print(f"Loaded in {time.time() - start:.1f}s")
print(f"  source1: {len(source1):,} rows")
print(f"  source2: {len(source2):,} rows")
print(f"  source3: {len(source3):,} rows")
print(f"  ground_truth: {len(ground_truth):,} rows")

# Build the ground-truth lookup once: source1_entity_id -> set of matched ids
gt_lookup = {}

for row in ground_truth.itertuples(index=False):
    value = row.matched_entity_ids

    if pd.isna(value) or str(value).strip() == "":
        matches = set()
    else:
        matches = {x.strip() for x in str(value).split(",") if x.strip()}

    gt_lookup[row.source1_entity_id] = matches

entities_with_matches = sum(1 for v in gt_lookup.values() if v)
print(f"\nEntities with at least one match: {entities_with_matches:,} / {len(gt_lookup):,}")

Loading datasets from parquet...
Loaded in 3.3s
  source1: 2,206,821 rows
  source2: 5,034,616 rows
  source3: 5,285,603 rows
  ground_truth: 2,206,821 rows

Entities with at least one match: 2,083,574 / 2,206,821
